# DP2 — Static Object / Source catalog query over a DDF via Butler

**Author:** dagoret  
**Creation Date:** 2026-03-16  
**Last UpDate:** 2026-03-18 : add selection Isolated Star, (not extended and primary)   <==> drop patch selection
**version:** v1
## Purpose

Retrieve **Object**, **Source**, and **ForcedSource** catalogs
for a user-selected Deep Drilling Field (DDF) using **Butler Gen3 only**
(no TAP service, not yet available for DP2 at USDF).

## Actual schema (discovered at runtime — COSMOS, LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545)

- `object` : dims `(skymap, tract)` — 1 ref per tract, index = `ObjectId`
- `source` : dims `(skymap, tract)` — 1 ref per tract, index = `SourceId` (join on `ObjectId` column)
- `object_forced_source` : dims `(skymap, tract, patch)` — **21 refs per tract** (one per patch)
  - Must iterate over all patch refs and concat
  - Contains per-visit forced photometry linked to DiaObjects

## Reference notebooks
- `2026-03-07_AccessToDP2.ipynb` — Butler setup
- `2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb` — tract lookup

---
## 0. Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import traceback
import gc


import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import Angle

from scipy.stats import median_abs_deviation

from lsst.daf.butler import Butler
from lsst.geom import SpherePoint, degrees

print(f"Matplotlib : {matplotlib.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

In [ ]:
mpl.rcParams.update(
    {
        "figure.figsize": (8, 5),  # default figure size
        "font.size": 14,  # taille de base
        "axes.titlesize": 18,  # titre de l'axe
        "axes.labelsize": 16,  # labels x et y
        "xtick.labelsize": 14,  # ticks x
        "ytick.labelsize": 14,  # ticks y
        "legend.fontsize": 14,  # legend text
        "legend.title_fontsize": 15,  # legend title
        "figure.titlesize": 20,  # titre global de la figure
    }
)

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
# try:
#    import ipympl  # noqa: F401
#    %matplotlib widget
#    print("ipympl found → interactive backend (%matplotlib widget)")
# except ImportError:
#    %matplotlib inline
#    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
#    print("Install with:  pip install ipympl")


In [ ]:
# output dirs
NB_TAG = "DP2_DDF_STATICOBJ_BUTLER_01"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")


# Output directory for DRP data
OUTPUT_DIR = "data_DP2_DDF_OBJECT_DRP_01"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Visits filename
FILENAME_VISITS = "visitTable-2025041500138-2026010800515_N34276.csv"

In [ ]:
# debug flags
DEBUG = False
DEBUG_VISITS = False

In [ ]:
# FLAGS
FLAG_FETCH_VISITSEXPOSURES = False

## Usefull functions

- from https://github.com/sylvielsstfr/mySIT-COM2025/blob/main/notebooks/2025-05-30-Visits/05_SourcesFromVisits.ipynb

In [ ]:
def getLostOfVisits(registry, where_clause_date):
    """
    Get exposure/visits from butler registry
    parameters:
    ==========
        registry : butler registry
        where_clause_date : specify which instrument and exposure dates in the butler registry
    """
    df_exposure = pd.DataFrame(
        {
            "id": pd.Series(dtype="int"),
            "obs_id": pd.Series(dtype="int"),
            "day_obs": pd.Series(dtype="int"),
            "seq_num": pd.Series(dtype="int"),
            "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c'est un datetime
            "time_end": pd.Series(dtype="str"),  # idem
            "type": pd.Series(dtype="str"),
            "target": pd.Series(dtype="str"),
            "filter": pd.Series(dtype="str"),
            "zenith_angle": pd.Series(dtype="float"),
            "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
            "ra": pd.Series(dtype="float"),
            "dec": pd.Series(dtype="float"),
            "skyangle": pd.Series(dtype="float"),
            "azimuth": pd.Series(dtype="float"),
            "zenith": pd.Series(dtype="float"),
            "science_program": pd.Series(dtype="str"),
            "jd": pd.Series(dtype="float"),
            "mjd": pd.Series(dtype="float"),
        }
    )

    # save the data array in rows before saving in pandas dataframe
    rows = []
    for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
        try:
            jd_start = info.timespan.begin.value
            jd_end = info.timespan.end.value
            the_Time_start = Time(jd_start, format="jd", scale="utc")
            the_Time_end = Time(jd_end, format="jd", scale="utc")
            mjd_start = the_Time_start.mjd
            mjd_end = the_Time_end.mjd
            isot_start = the_Time_start.isot
            isot_end = the_Time_end.isot

            if count == 0:
                print("===== Time Conversion Debug Info =====")
                print(f"JD start      : {jd_start} (type: {type(jd_start)})")
                print(f"JD end        : {jd_end} (type: {type(jd_end)})")
                print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
                print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
                print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
                print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
                print("=======================================")

            # put row in a dictionnary before stacking
            row = {
                "id": info.id,
                "obs_id": info.obs_id,
                "day_obs": info.day_obs,
                "seq_num": info.seq_num,
                "time_start": isot_start,
                "time_end": isot_end,
                "type": info.observation_type,
                "target": info.target_name,
                "filter": info.physical_filter,
                "zenith_angle": info.zenith_angle,
                "expos": info.exposure_time,  # Exemple : adapter selon ton objet
                "ra": info.tracking_ra,
                "dec": info.tracking_dec,
                "skyangle": info.sky_angle,
                "azimuth": info.azimuth,
                "zenith": info.zenith_angle,
                "science_program": info.science_program,
                "jd": float(jd_start),
                "mjd": float(mjd_start),
            }
            rows.append(row)

        except ValueError as e:
            print(f"Erreur de valeur : {e}")
        except FileNotFoundError as e:
            print(f"Fichier introuvable : {e}")
        except Exception as e:
            print(f"Erreur inattendue : {type(e).__name__} - {e}")
            print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
            traceback.print_exc()  # displays the full stack trace

    # Final DataFrame creation
    df_exposure = pd.DataFrame(rows)
    df_science = df_exposure[df_exposure.type == "science"]
    df_science.reset_index(drop=True, inplace=True)

    return df_science

In [ ]:
def FetchTimesForVisits(visit_list):
    """
    Keep for example and possible later use
    """
    # On interroge la table visitDefinition
    Nvisit = len(visit_list)

    if Nvisit == 1:
        thevisit = visit_list.values[0]
        rows = registry.queryDimensionRecords("visit", where=f"visit={thevisit}")
    else:
        rows = registry.queryDimensionRecords("visit", where=f"visit in {tuple(visit_list)}")

    # 4. Build a results table
    results = []
    for row in rows:
        visit_id = row.id
        visit_airmass = 1.0 / np.cos(Angle(row.zenith_angle, u.degree).rad)
        visit_azimuth = row.azimuth

        # Extraire l'instant de début de l'observation (Time astropy)
        start_time = row.timespan.begin

        # Convertir en MJD et ISO
        mjd = start_time.to_value("mjd")  # Ex: 60384.28718
        isot = start_time.to_value("isot")  # Ex: '2024-04-19 06:53:32.000'

        # mjd = row.startDate.toMjd()
        # utc = Time(mjd, format='mjd', scale='utc').to_value('iso')
        # results.append({"visit": visit_id, "mjd": mjd, "isot": isot})
        results.append(
            {"visit": visit_id, "mjd": mjd, "isot": isot, "airmass": visit_airmass, "azimuth": visit_azimuth}
        )

    df_times = pd.DataFrame(results).sort_values("visit")
    df_times.set_index("visit", inplace=True)
    return df_times

In [ ]:
def RetrieveDRPSources_forTarget(butler, center_coord, datasettype, where_clause, radius_cut=50):
    """
    Keep for example and possible later use
    Find the closest Sourcesthe target_coord

    parameters:
    - butler
    - the coordinate of the target (center of the cone seach)
    - the datasettype name for the DRP object
    - where_clause : which contrain requirements on the tract and patch numbers
    - cut on angluar separation for the returned for the returned object

    Return
    - object Id with minimum separation ,
    - minimum separation (arcec),
    - the table of DRP objects within the radius_cut
    """

    ra_columns = ["u_ra", "g_ra", "r_ra", "i_ra", "z_ra", "y_ra"]
    dec_columns = ["u_dec", "g_dec", "r_dec", "i_dec", "z_dec", "y_dec"]

    therefs = butler.registry.queryDatasets(datasettype, collections=collection, where=where_clause)

    for count, ref in enumerate(therefs):
        the_id = ref.dataId
        the_tract_id = the_id["tract"]
        print(the_id)

        # catalog of rubin objects (a pandas Dataframe) inside the tract
        catalog = butler.get(ref)
        catalog = catalog[catalog["patch"] == patchNbSel]

        nobjects = len(catalog)

        # Calcul de la moyenne ligne par ligne, en ignorant les NaN
        catalog["ra"] = catalog[ra_columns].mean(axis=1, skipna=True)
        catalog["dec"] = catalog[dec_columns].mean(axis=1, skipna=True)

        # extract the (ra,dec) coordinates for all te objects of the rubin-catalog
        ra_cat = catalog["ra"].values
        dec_cat = catalog["dec"].values
        # coordinates for all rubin-catalog points
        catalog_coords = SkyCoord(ra=ra_cat * u.deg, dec=dec_cat * u.deg)

        # Angular distance to target
        distances_arcsec = center_coord.separation(catalog_coords).arcsecond

        # add the separation angle to the ctalog
        catalog["sep"] = distances_arcsec

        # closest object from the target
        sepMin = distances_arcsec.min()
        sepMin_idx = np.where(distances_arcsec == sepMin)[0][0]

        closest_obj = catalog[catalog["sep"] <= sepMin]

        # select a few of these sources to debug the closest candidate
        nearby_obj = catalog[distances_arcsec < radius_cut]

        return closest_obj, sepMin, nearby_obj

In [ ]:
# =========================================================================
# Helper: add a top x-axis showing calendar dates above the MJD axis
# =========================================================================
# The bottom x-axis of each light curve panel uses MJD.
# This helper adds a twin axis on top with evenly-spaced date labels
# (UTC) so you can immediately read off known observing dates.
#
# Usage (call AFTER plotting, BEFORE tight_layout / savefig):
#   add_date_top_axis(axes[0], n_ticks=6)
# =========================================================================


def add_date_top_axis(ax, n_ticks=6, date_fmt="%Y-%m-%d"):
    """
    Add a secondary x-axis on top of `ax` showing calendar dates.

    The secondary axis shares the same MJD limits as the primary axis
    but labels ticks as UTC calendar dates (e.g. 2025-06-15).

    Parameters
    ----------
    ax : matplotlib Axes
        Primary axes whose bottom x-axis carries MJD values.
    n_ticks : int
        Number of evenly-spaced tick marks on the top axis.
    date_fmt : str
        strftime format for the date labels (default YYYY-MM-DD).

    Returns
    -------
    ax_top : matplotlib Axes
        The new twin axes placed on top.
    """
    mjd_min, mjd_max = ax.get_xlim()

    # Build evenly-spaced MJD tick positions spanning the current x range
    mjd_ticks = np.linspace(mjd_min, mjd_max, n_ticks)

    # Convert each MJD tick to a calendar date string via astropy
    date_labels = [Time(m, format="mjd", scale="utc").to_value("isot")[:10] for m in mjd_ticks]

    # Create a twin axis that shares the same x-scale
    ax_top = ax.twiny()
    ax_top.set_xlim(mjd_min, mjd_max)  # must match primary axis exactly
    ax_top.set_xticks(mjd_ticks)
    ax_top.set_xticklabels(date_labels, rotation=30, ha="left", fontsize=8)
    ax_top.set_xlabel("Date (UTC)", fontsize=9, labelpad=4)

    return ax_top


print("add_date_top_axis() helper defined.")

In [ ]:
def load_visits_file(file_path):
    """
    Checks if the visits file exists and loads it into a DataFrame.
    """
    if not os.path.exists(file_path):
        print(f"ERROR: File not found at {file_path}")
        return None

    try:
        # Standard LSST tables are usually Parquet
        if file_path.endswith(".parquet"):
            df_visits = pd.read_parquet(file_path)
        else:
            df_visits = pd.read_csv(file_path)

        print(f"Successfully loaded {len(df_visits):,} visits from {os.path.basename(file_path)}")
        return df_visits
    except Exception as e:
        print(f"ERROR loading file: {e}")
        return None


# --- Usage ---
# df_visits = load_visits_file(FULLFILENAME_VISITS)

---
## 1. User Configuration

In [ ]:
SELECTED_DDF = "COSMOS"  # COSMOS | ECDFS | ELAIS-S1 | XMM-LSS | EDFS-a | EDFS-b | EDFS | M49
# Check the selected TRACT and PATCH exist for the selected DDF
SELECTED_TRACT = 9813
# SELECTED_PATCH = 83
# SELECTED_PATCH = 58
CONE_RADIUS_DEG = 1.8
MIN_NSOURCES = 500

REPO = "dp2_prep"
COLLECTION = "LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545"
SKYMAP_NAME = "lsst_cells_v2"
INSTRUMENT = "LSSTCam"
WHERE_CLAUSE_INSTRUMENT = "instrument = '" + f"{INSTRUMENT}" + "'"
DATE_START = 20250415
WHERE_CLAUSE_DATE = WHERE_CLAUSE_INSTRUMENT + f" and day_obs >= {DATE_START}"

TRACT_SEARCH_RADIUS_DEG = 1.8

DDF_COORDS = {
    "COSMOS": (150.119, +2.206),
    "ECDFS": (53.125, -28.100),
    "ELAIS-S1": (9.450, -44.000),
    "XMM-LSS": (35.708, -4.750),
    "EDFS-a": (58.900, -49.315),
    "EDFS-b": (63.600, -47.600),
    "EDFS": (61.240, -48.423),
    "M49": (187.400, +8.000),
}

RA_CENTER, DEC_CENTER = DDF_COORDS[SELECTED_DDF]
print(f"DDF         : {SELECTED_DDF}  ({RA_CENTER:.4f}°, {DEC_CENTER:+.4f}°)")
print(f"Cone radius : {CONE_RADIUS_DEG}°")
print(f"Min nSrc : {MIN_NSOURCES}")
print(f"Collection  : {COLLECTION}")
print(f"Instrument  : {INSTRUMENT}")
print(f"Where clause  : {WHERE_CLAUSE_INSTRUMENT}")
print(f"Where clause date : {WHERE_CLAUSE_DATE}")

---
## 2. Butler and SkyMap

In [ ]:
butler = Butler(REPO, collections=COLLECTION)
registry = butler.registry
print("Butler OK")

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)
print(f"SkyMap '{SKYMAP_NAME}' : {len(skymap)} tracts")

### 2.1 Search for visit tables

In [ ]:
visitTable_pattern1 = "*isitTable*"
visitTable_pattern2 = "*isitTable"
visitTable_name = "preVisitTable"

In [ ]:
if DEBUG_VISITS:
    dataset_types = registry.queryDatasetTypes(visitTable_pattern1)
    for dt in dataset_types:
        print(dt.name)

In [ ]:
if DEBUG_VISITS:
    dataset_types = list(butler.registry.queryDatasetTypes(visitTable_pattern1))

    # Pour un affichage lisible (nom du type et storage class)
    for dt in sorted(dataset_types, key=lambda x: x.name):
        print(f"{dt.name:20} | {dt.storageClass.name}")

In [ ]:
if DEBUG_VISITS:
    dataset_types = list(butler.registry.queryDatasetTypes(visitTable_pattern2))
    for dt in dataset_types:
        print(f"{dt.name} :::", dt)
        print(f"    required dimensions: {dt.dimensions.required}")
        print()

## 2.2 Retrieve the science visits in order to match the visitid with the MJD in forced source photometry
- `visitid = id`
- `MJD = mjd `

In [ ]:
if FLAG_FETCH_VISITSEXPOSURES:
    visitTable = getLostOfVisits(registry, where_clause_date=WHERE_CLAUSE_DATE)
    print(visitTable.head(1))
    visitMin, visitMax, Nvisits = visitTable["id"].min(), visitTable["id"].max(), len(visitTable)
    print(f"visitMin = {visitMin},visitMax = {visitMax}, Nvisits = {Nvisits}")
    visit_filename = f"visitTable-{visitMin}-{visitMax}_N{Nvisits}.csv"
    print(f"filename_visits = {visit_filename}")
    visit_fullfilename = os.path.join(OUTPUT_DIR, visit_filename)
    visitTable.to_csv(visit_fullfilename)
    del visitTable
    gc.collect()

---
## 3. Dataset Type Inventory

Identify the correct names and **dimensions** for DIA products — in particular
`dia_object_forced_source` has `(skymap, tract, patch)` dimensions, which
requires iterating over patch refs.

In [ ]:
# Hard-coded after discovery: confirmed present in DM-54249
DATASET_TYPE_OBJ = "object"  # dims: skymap, tract
DATASET_TYPE_SRC = "source"  # dims: skymap, tract
DATASET_TYPE_FORCED = "object_forced_source"  # dims: skymap, tract, patch


# Print actual dimensions for confirmation
for dt_name in [DATASET_TYPE_OBJ, DATASET_TYPE_SRC, DATASET_TYPE_FORCED]:
    try:
        dt = registry.getDatasetType(dt_name)
        in_col = registry.queryDatasets(dt_name, collections=COLLECTION).any(execute=False, exact=False)
        print(f"{dt_name:40s}  dims={list(dt.dimensions.names)}  present={in_col}")
    except Exception as exc:
        print(f"{dt_name:40s}  ERROR: {exc}")

---
## 4. Find Tracts Covering the DDF

In [ ]:
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=1.8):
    cos_dec = max(np.cos(np.deg2rad(dec_deg)), 0.01)
    step = 0.35
    found_ids = set()
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s = ra_deg + dra / cos_dec
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = SpherePoint(ra_s * degrees, dec_s * degrees)
            try:
                found_ids.add(skymap.findTract(sp).tract_id)
            except Exception:
                pass
    return sorted(found_ids)


tract_ids = find_tracts_for_coord(skymap, RA_CENTER, DEC_CENTER, radius_deg=TRACT_SEARCH_RADIUS_DEG)
print(f"Tracts covering {SELECTED_DDF}: {tract_ids}")

---
## 5. Schema Discovery (one-time probe)

### 5.1 Object Schema

#### Probe DATASET_TYPE_OBJ 

In [ ]:
refs_probe = list(
    registry.queryDatasets(
        DATASET_TYPE_OBJ,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
    )
)
assert refs_probe, "No  object ref found."

obj_raw = butler.get(refs_probe[0])
df_probe = obj_raw.to_pandas() if hasattr(obj_raw, "to_pandas") else obj_raw

print(f"index.name  : {df_probe.index.name!r}")
print(f"index.dtype : {df_probe.index.dtype}")
print(f"index[:3]   : {df_probe.index[:3].tolist()}")
print(f"columns     : {list(df_probe.columns)[:50]}")

#### Probe OBJECT ID name

In [ ]:
# Determine ID column
if df_probe.index.name is not None:
    OBJ_ID_COL = df_probe.index.name
    ID_IN_INDEX = True
    print(f"Object ID in index: '{OBJ_ID_COL}'")
elif any(c in df_probe.columns for c in ["objectId", "object_id"]):
    OBJ_ID_COL = next(c for c in ["objectId", "object_id"] if c in df_probe.columns)
    ID_IN_INDEX = False
    print(f"Object ID as column: '{OBJ_ID_COL}'")
else:
    OBJ_ID_COL = "row_id"
    ID_IN_INDEX = False
    print("WARNING: no ID found, using row index.")

#### Probe existence of DATASET_TYPE_FORCED

In [ ]:
# Also probe object_forced_source schema (patch-level)
refs_forced_probe = list(
    registry.queryDatasets(
        DATASET_TYPE_FORCED,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
    )
)
print(f"object_forced_source refs for tract {tract_ids[0]}: {len(refs_forced_probe)}")

if refs_forced_probe:
    frc_raw = butler.get(refs_forced_probe[0])
    df_fprobe = frc_raw.to_pandas() if hasattr(frc_raw, "to_pandas") else frc_raw
    print(f"index.name  : {df_fprobe.index.name!r}")
    print(f"columns     : {list(df_fprobe.columns)}")
    print(f"shape       : {df_fprobe.shape}")

---
## 6. Helper: `to_dataframe()`

In [ ]:
def to_dataframe(obj, id_col=None, id_in_index=None):
    """
    Convert Butler catalog to pandas DataFrame.
    If id_in_index is True, promotes the named index to a regular column.
    Falls back to global OBJ_ID_COL / ID_IN_INDEX if not provided.
    """
    _id_col = id_col if id_col is not None else OBJ_ID_COL
    _id_idx = id_in_index if id_in_index is not None else ID_IN_INDEX

    if isinstance(obj, pd.DataFrame):
        df = obj
    elif hasattr(obj, "to_pandas"):
        df = obj.to_pandas()
    elif hasattr(obj, "to_table"):
        df = obj.to_table().to_pandas()
    elif isinstance(obj, Table):
        df = obj.to_pandas()
    else:
        raise TypeError(f"Cannot convert {type(obj)} to DataFrame.")

    if _id_idx and _id_col not in df.columns:
        df = df.copy()
        df.insert(0, _id_col, df.index)
        df = df.reset_index(drop=True)
    elif _id_col == "row_id" and _id_col not in df.columns:
        df.insert(0, _id_col, range(len(df)))

    return df

---
## 7. Load Object Catalog

### 7.1 Select only one TRACT

In [ ]:
tract_ids = [SELECTED_TRACT]

### 7.2 Retrieve Objects from the Butler

In [ ]:
obj_frames = []

# loop on tracts
for tract_id in tract_ids:
    refs = list(
        registry.queryDatasets(
            DATASET_TYPE_OBJ, collections=COLLECTION, dataId={"skymap": SKYMAP_NAME, "tract": tract_id}
        )
    )
    for ref in refs:
        try:
            df_tmp = to_dataframe(butler.get(ref))
            df_tmp["_tract"] = tract_id
            obj_frames.append(df_tmp)
            print(f"  Tract {tract_id:6d} — {len(df_tmp):,} Objects")
        except Exception as exc:
            print(f"  Tract {tract_id:6d} — ERROR: {exc}")

assert obj_frames, "No object data loaded."
df_obj_all = pd.concat(obj_frames, ignore_index=True)
n_before = len(df_obj_all)
df_obj_all = df_obj_all.drop_duplicates(subset=OBJ_ID_COL)
gc.collect()
n_after = len(df_obj_all)
print(f"\nTotal: {n_before:,} → {n_after:,} after dedup")
gc.collect()

#### Select Object points 

In [ ]:
mask1 = df_obj_all["griz_model_extendedness"] == 0
mask2 = df_obj_all["detect_isIsolated"] == True
mask3 = df_obj_all["parentObjectId"] == 0
mask = mask1 & mask2 & mask3
df_obj_all = df_obj_all[mask]
n_filtered = len(df_obj_all)
print(f"\nTotal after filtering: {n_after:,} → {n_filtered:,} after selection")
gc.collect()

### 7.3 Detect column name in object Table

In [ ]:
# --- 1. Detect ID Column ---
OBJ_ID_COL = next((c for c in ["objectId", "id"] if c in df_obj_all.columns), None)

# --- 2. Detect RA/DEC Columns ---
# In DRP, these are often coord_ra/coord_dec (in radians)
# but once converted to pandas they might be ra/dec
# Detect column names
RA_COL, DEC_COL = next(
    (
        (ra, dec)
        for ra, dec in [("ra", "dec"), ("coord_ra", "coord_dec"), ("RA", "DEC")]
        if ra in df_obj_all.columns
    ),
    (None, None),
)
assert RA_COL, f"No RA/Dec found. Columns: {list(df_obj_all.columns)}"

# --- 3. Detect Source Count (Memory Trick) ---
# DRP Object tables don't have a single 'nSources' column.
# We use '{band}_inputCount' instead.
# Let's find the first available band to use as a proxy or sum them.
# NSRC_COL = next((c for c in ['nSources','n_sources'] if c in df_obj_all.columns), None)
input_count_cols = [c for c in df_obj_all.columns if c.endswith("_inputCount")]

if input_count_cols:
    # Use the first band found (e.g., 'i_inputCount') as the NSRC_COL for filtering
    NSRC_COL = input_count_cols[0]
    print(f"Using '{NSRC_COL}' as proxy for observation count.")
else:
    NSRC_COL = None

# --- 4. Detect Bands ---
# We look for prefixes like 'g_', 'r_', etc.
target_order = ["u", "g", "r", "i", "z", "y"]
BANDS = sorted(
    {c.split("_")[0] for c in df_obj_all.columns if "_" in c and c.split("_")[0] in target_order},
    key=lambda b: target_order.index(b),
)
BAND_COLORS = {"u": "purple", "g": "green", "r": "red", "i": "darkorange", "z": "sienna", "y": "black"}

print(f"ID   : '{OBJ_ID_COL}'")
print(f"RA   : '{RA_COL}'  Dec: '{DEC_COL}'")
print(f"nSrc Proxy : '{NSRC_COL}'")
print(f"Bands: {BANDS}")

### 7.4 Select object in a cone

In [ ]:
# Spatial cone cut
center_sky = SkyCoord(ra=RA_CENTER * u.deg, dec=DEC_CENTER * u.deg)
sep_deg = center_sky.separation(
    SkyCoord(ra=df_obj_all[RA_COL].values * u.deg, dec=df_obj_all[DEC_COL].values * u.deg)
).deg

df_obj_cone = df_obj_all[sep_deg <= CONE_RADIUS_DEG].copy()
df_obj_cone["_sep_deg"] = sep_deg[sep_deg <= CONE_RADIUS_DEG]
print(f"In cone: {len(df_obj_cone):,} / {len(df_obj_all):,}")

### 7.5 Add Missing columns to Object table for later selection

- add `total_obs_count`

In [ ]:
# --- 1. Create a proxy for nSources ---
# In DRP, we sum the 'inputCount' across all available bands to get the total number of observations
input_count_cols = [c for c in df_obj_all.columns if c.endswith("_inputCount")]

if input_count_cols:
    # Calculate a temporary total count to mimic MIN_NSOURCES behavior
    df_obj_all["total_obs_count"] = df_obj_all[input_count_cols].sum(axis=1)
    NSRC_COL = "total_obs_count"
else:
    NSRC_COL = None

# --- 2. Filter and Sort ---
if NSRC_COL and NSRC_COL in df_obj_all.columns:
    # Filter objects that have enough detections across all bands
    df_obj_rich = (
        df_obj_all[df_obj_all[NSRC_COL] >= MIN_NSOURCES]
        .sort_values(NSRC_COL, ascending=False)
        .reset_index(drop=True)
    )
    print(
        f"Filtered: {len(df_obj_rich):,} objects "
        f"(max detections={df_obj_rich[NSRC_COL].max()}, "
        f"median={df_obj_rich[NSRC_COL].median():.1f})"
    )
else:
    df_obj_rich = df_obj_all.copy()
    print("Warning: No observation count column found — keeping all objects.")

# --- 3. Clean up Memory ---
# If df_obj_all is very large, delete it after filtering to free up RAM
# del df_obj_all
# import gc
# gc.collect()

# --- 4. Display Results ---
# Adapt peek_cols for DRP naming (tract and patch usually don't have underscores in DRP)
peek_cols = [c for c in [OBJ_ID_COL, RA_COL, DEC_COL, NSRC_COL, "tract", "patch"] if c in df_obj_rich.columns]

display(df_obj_rich[peek_cols].head(10))

---
## 8. Diagnostic Plots — Object

Changement de logique (Histogramme de droite) : Au lieu d'afficher simplement le catalogue "filtré", j'ai remplacé le second graphe par une comparaison des bandes. En DRP, il est très utile de voir si une bande (souvent i ou r) a beaucoup plus d'observations que les autres (comme u ou y).

histtype='step' : Pour le graphe multi-bandes, le type step évite que les barres ne se cachent les unes les autres, ce qui arrive souvent quand on a 6 bandes superposées.

Échelle logarithmique (log=True) : Indispensable en DRP car le nombre d'objets avec peu d'observations est massivement plus élevé que les objets bien échantillonnés.

Gestion des couleurs : Utilisation du dictionnaire BAND_COLORS pour maintenir une cohérence visuelle avec les courbes de lumière que vous ferez plus tard.

### 8.1 Remove unsued columns

In [ ]:
# List only what you actually use
essential_cols = [OBJ_ID_COL, RA_COL, DEC_COL, "tract", "patch"]
# Add the inputCounts for all bands
essential_cols += [f"{b}_inputCount" for b in BANDS]
# Add the flux columns for later lightcurve selection if needed
essential_cols += [f"{b}_psfFlux" for b in BANDS]
essential_cols += [f"{b}_psfFluxErr" for b in BANDS]
essential_cols += ["total_obs_count"]

In [ ]:
df_obj_all = df_obj_all[essential_cols]
df_obj_rich = df_obj_rich[essential_cols]
gc.collect()

###  8.2 Plot Count Distribution

In [ ]:
# --- Configuration for the per-band plot ---
# Using the list of bands detected earlier (e.g., ['u', 'g', 'r', 'i', 'z', 'y'])
# and the colors defined in BAND_COLORS
n_bands = len(BANDS)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left Plot: Distribution of Total Observations (Sum across bands) ---
ax = axes[0]
if NSRC_COL in df_obj_all.columns:
    # Plotting the total sum of inputCounts (calculated in the previous step)
    ax.hist(
        df_obj_all[NSRC_COL].dropna(),
        bins=100,
        log=True,
        color="steelblue",
        alpha=0.8,
        edgecolor="white",
        linewidth=0.3,
    )
    ax.axvline(MIN_NSOURCES, color="red", ls="--", lw=1.5, label=f"cut >= {MIN_NSOURCES}")

ax.set(
    xlabel="Total inputCount (All bands)",
    ylabel="N Objects",
    title=f"{SELECTED_DDF} — DRP Objects in cone (N={len(df_obj_all):,})",
)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# --- Right Plot: Distribution per band ---
ax = axes[1]
for band in BANDS:
    col = f"{band}_inputCount"
    if col in df_obj_all.columns:
        # We plot the distribution for each band with its specific color
        # We use a step plot or a histogram with high transparency to see overlaps
        counts = df_obj_all[col].dropna()
        if len(counts) > 0:
            ax.hist(
                counts,
                bins=50,
                histtype="step",
                lw=2,
                label=f"band {band}",
                color=BAND_COLORS.get(band, "black"),
                alpha=0.9,
            )

ax.set_yscale("log")  # Usually better for DRP counts
ax.set(xlabel="inputCount per band", ylabel="N Objects", title=f"{SELECTED_DDF} — Observations per band")
ax.legend(loc="upper right", ncol=2, fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save the figure with a DRP specific name
fig_name = f"DRP_Obj_inputCount_{SELECTED_DDF}.png"
plt.savefig(f"{DIR_FIGS}/{fig_name}", dpi=150, bbox_inches="tight")
plt.show()

###  8.3 Plot Count Map

In [ ]:
# --- 1. Prepare the figure ---
fig, ax = plt.subplots(figsize=(10, 9))
ax.set_facecolor("#f5f5f5")
ax.invert_xaxis()  # Standard for Sky Maps (RA increases to the left)

# --- 2. Plot all objects found in the cone (Background) ---
ax.scatter(
    df_obj_all[RA_COL],
    df_obj_all[DEC_COL],
    s=1,
    color="lightgrey",
    alpha=0.4,
    zorder=1,
    label=f"All DRP Objects ({len(df_obj_all):,})",
)

# --- 3. Plot "Rich" objects (Foreground with colormap) ---
# NSRC_COL here is our proxy (total_obs_count) calculated from inputCounts
if NSRC_COL and len(df_obj_rich) > 0:
    # Use a LogNorm because the number of observations varies significantly
    norm_min = max(1, df_obj_rich[NSRC_COL].min())  # Ensure min > 0 for LogNorm
    norm_max = df_obj_rich[NSRC_COL].max()

    sc = ax.scatter(
        df_obj_rich[RA_COL],
        df_obj_rich[DEC_COL],
        c=df_obj_rich[NSRC_COL],
        cmap="plasma",
        s=14,
        alpha=0.9,
        zorder=3,
        norm=mcolors.LogNorm(vmin=norm_min, vmax=norm_max),
        label=f"Total Obs >= {MIN_NSOURCES} ({len(df_obj_rich):,})",
    )

    cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
    cbar.set_label("Total inputCount (all bands)", fontsize=11)

# --- 4. Plot Search Center and Cone Boundary ---
ax.plot(
    RA_CENTER,
    DEC_CENTER,
    marker="*",
    ms=18,
    color="gold",
    mec="black",
    mew=1,
    zorder=10,
    ls="none",
    label=f"{SELECTED_DDF} Center",
)

# Projection correction for the circle
cos_dec = np.cos(np.deg2rad(DEC_CENTER))
theta = np.linspace(0, 2 * np.pi, 361)
ax.plot(
    RA_CENTER + CONE_RADIUS_DEG / cos_dec * np.cos(theta),
    DEC_CENTER + CONE_RADIUS_DEG * np.sin(theta),
    "r--",
    lw=1.2,
    zorder=4,
    label=f"Cone {CONE_RADIUS_DEG} deg",
)

# --- 5. Axes and Labels ---
ax.set_xlabel("RA (deg)", fontsize=12)
ax.set_ylabel("Dec (deg)", fontsize=12)
ax.set_title(f"DP2 DRP Objects — Sky Distribution — {SELECTED_DDF}\n(Tracts {tract_ids})", fontsize=14)

ax.legend(fontsize=10, loc="upper right", frameon=True, framealpha=0.9)
ax.grid(True, alpha=0.3, ls="--")

plt.tight_layout()

# --- 6. Save Figure ---
fig_name = f"DRP_Obj_sky_{SELECTED_DDF}.png"
plt.savefig(f"{DIR_FIGS}/{fig_name}", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
if 0:
    # --- 1. Setup the 2x3 grid ---
    fig, axes = plt.subplots(2, 3, figsize=(18, 12), sharex=True, sharey=True)
    axes = axes.flatten()  # Flatten to 1D array for easier iteration

    # Using the standard Rubin band order
    target_order = ["u", "g", "r", "i", "z", "y"]

    # --- 2. Iterate through bands and plot ---
    for idx, band in enumerate(target_order):
        ax = axes[idx]
        col_count = f"{band}_inputCount"

        # Plot the background (all objects in the cone for context)
        ax.scatter(df_obj_all[RA_COL], df_obj_all[DEC_COL], s=0.5, color="lightgrey", alpha=0.3, zorder=1)

        if col_count in df_obj_all.columns:
            # Filter objects that have at least 1 detection in THIS specific band
            # We also highlight the "rich" objects based on the total count
            # but color them by the specific band's count
            mask_band = df_obj_rich[col_count] > 0
            df_plot = df_obj_rich[mask_band]

            if len(df_plot) > 0:
                sc = ax.scatter(
                    df_plot[RA_COL],
                    df_plot[DEC_COL],
                    c=df_plot[col_count],
                    cmap="viridis",  # Different cmap to distinguish from the global plot
                    s=8,
                    alpha=0.8,
                    zorder=3,
                    norm=mcolors.LogNorm(vmin=1, vmax=df_obj_rich[col_count].max()),
                )

                # Add a small colorbar for each subplot to see band-specific depth
                cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
                cbar.set_label(f"{band} inputCount", fontsize=9)

        # Decoration
        ax.set_title(
            f"Filter: {band.upper()}", fontsize=14, fontweight="bold", color=BAND_COLORS.get(band, "black")
        )
        ax.set_facecolor("#fcfcfc")
        ax.invert_xaxis()
        ax.grid(True, alpha=0.2, ls="--")

        if idx >= 3:
            ax.set_xlabel("RA (deg)", fontsize=11)
        if idx % 3 == 0:
            ax.set_ylabel("Dec (deg)", fontsize=11)

    # --- 3. Global settings & Save ---
    fig.suptitle(
        f"DRP Object Distribution per Band — {SELECTED_DDF}\n(Objects with Total Obs >= {MIN_NSOURCES})",
        fontsize=18,
        y=0.95,
    )

    plt.tight_layout(rect=[0, 0.03, 1, 0.93])

    fig_name = f"DRP_Obj_sky_per_band_{SELECTED_DDF}.png"
    plt.savefig(f"{DIR_FIGS}/{fig_name}", dpi=150, bbox_inches="tight")
    plt.show()

## 9.0 Vizualisation on objects

### 9.1 Plot Histogram per patches

In [ ]:
# --- 1. Calculate the number of objects per patch ---
# We group by 'patch' and count the number of objectId entries
df_patch_counts = df_obj_rich.groupby("patch").size().reset_index(name="n_objects")
df_patch_counts = df_patch_counts.sort_values("n_objects", ascending=False).reset_index(drop=True)

# Print summary statistics
print(f"Total number of patches in tract: {len(df_patch_counts)}")
print(f"Total objects in filtered catalog: {df_patch_counts['n_objects'].sum():,}")
print(f"Average density: {df_patch_counts['n_objects'].mean():.1f} objects/patch")

# --- 2. Create the Visualizations ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot A: Histogram (Frequency of densities)
ax = axes[0]
ax.hist(df_patch_counts["n_objects"], bins=20, color="skyblue", edgecolor="white", alpha=0.8)
ax.set_xlabel("Number of Objects in a Patch", fontsize=12)
ax.set_ylabel("Frequency (Number of Patches)", fontsize=12)
ax.set_title("Distribution of Object Density per Patch", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3)

# Plot B: Bar Chart (Top 50 most populated patches)
ax = axes[1]
top_n = 50
df_plot = df_patch_counts.head(top_n).copy()
# Convert patch to string to ensure categorical axis
df_plot["patch_str"] = df_plot["patch"].astype(str)

ax.bar(df_plot["patch_str"], df_plot["n_objects"], color="steelblue", alpha=0.8)
ax.set_xlabel("Patch ID", fontsize=12)
ax.set_ylabel("Object Count", fontsize=12)
ax.set_title(f"Top {top_n} Patches (Sorted by Count)", fontsize=14, fontweight="bold")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()

# Save the figure
fig_path = f"{DIR_FIGS}/DRP_patch_distribution_{SELECTED_DDF}.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

# --- 3. Display the top 10 patches table ---
print("\nTop 10 Patches by Object Count:")
display(df_patch_counts.head(10))

### 9.2 Plot Flux error vs Flux in the 6 bands

In [ ]:
# --- Configuration ---
# DRP Object tables contain coadd photometry. We map:
# Mean  -> psfFlux (the coadded flux)
# Sigma -> psfFluxErr (the uncertainty of the coadd measurement)
# Ndata -> inputCount (number of visits that contributed to the coadd)

TOP_N = min(10000, len(df_obj_rich))
df_top = df_obj_rich.head(TOP_N)

ncols = 3
nrows = int(np.ceil(len(BANDS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = np.array(axes).flatten()

for idx, band in enumerate(BANDS):
    ax = axes[idx]

    # DRP Column Mapping
    col_flux = f"{band}_psfFlux"
    col_err = f"{band}_psfFluxErr"
    col_nobs = f"{band}_inputCount"

    if col_flux not in df_top.columns:
        ax.set_visible(False)
        continue

    # Filter for valid measurements in this specific band
    valid = df_top[col_flux].notna() & (df_top[col_nobs] > 0)

    x = df_top.loc[valid, col_flux]
    y = df_top.loc[valid, col_err] if col_err in df_top.columns else np.zeros(valid.sum())
    c = df_top.loc[valid, col_nobs] if col_nobs in df_top.columns else "grey"

    # Scatter plot: Flux vs Error
    sc_ = ax.scatter(x, y, c=c, cmap="viridis", s=12, alpha=0.7)

    # Add colorbar for observation count (depth)
    cbar = plt.colorbar(sc_, ax=ax)
    cbar.set_label("inputCount (N Visits)", fontsize=9)

    ax.set_xlabel(f"{band} psfFlux (nJy)", fontsize=10)
    ax.set_ylabel(f"{band} psfFluxErr (nJy)", fontsize=10)
    ax.set_title(f"Band {band} (N={valid.sum():,})", fontsize=12, fontweight="bold")
    ax.title.set_color(BAND_COLORS.get(band, "black"))

    # Usually, flux vs error plots are clearer in Log-Log scale for DRP
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.2)

# Hide empty subplots
for idx in range(len(BANDS), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(
    f"{SELECTED_DDF} — DRP Coadd Flux vs Error per band (top {TOP_N} objects)",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout()

# Save with a DRP-specific filename
fig_name = f"DRP_Obj_flux_summary_{SELECTED_DDF}.png"
plt.savefig(f"{DIR_FIGS}/{fig_name}", dpi=150, bbox_inches="tight")
plt.show()

### 9.3 Plot Magnitude error vs Magnitude in the 6 bands

In [ ]:
ncols = 3
nrows = int(np.ceil(len(BANDS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = np.array(axes).flatten()

for idx, band in enumerate(BANDS):
    ax = axes[idx]

    col_flux = f"{band}_psfFlux"
    col_err = f"{band}_psfFluxErr"
    col_nobs = f"{band}_inputCount"

    if col_flux not in df_top.columns:
        ax.set_visible(False)
        continue

    # Filter for valid & positive flux (crucial for log/mag conversion)
    valid = (df_top[col_flux] > 0) & df_top[col_err].notna() & (df_top[col_nobs] > 0)

    flux = df_top.loc[valid, col_flux]
    flux_err = df_top.loc[valid, col_err]

    # 1. Convert nJy to AB Magnitude
    # Zero point for nJy to AB is 31.4
    mag = 31.4 - 2.5 * np.log10(flux)

    # 2. Convert Flux Error to Magnitude Error
    mag_err = 1.0857 * (flux_err / flux)

    c = df_top.loc[valid, col_nobs]

    # Scatter plot: Mag vs MagErr
    sc_ = ax.scatter(mag, mag_err, c=c, cmap="viridis", s=12, alpha=0.7)

    # Colorbar
    cbar = plt.colorbar(sc_, ax=ax)
    cbar.set_label("inputCount (N Visits)", fontsize=9)

    ax.set_xlabel(f"{band} psfMag (AB)", fontsize=10)
    ax.set_ylabel(f"{band} psfMagErr", fontsize=10)
    ax.set_title(f"Band {band} (N={len(mag):,})", fontsize=12, fontweight="bold")
    ax.title.set_color(BAND_COLORS.get(band, "black"))

    # Invert X-axis: Bright objects (small mag) to the left, faint to the right
    # ax.invert_xaxis()

    # Use log scale for the error axis as it often spans several orders of magnitude
    # ax.set_yscale('log')
    ax.grid(True, which="both", alpha=0.2)
    ax.set_xlim(15.0, 28.0)
    ax.set_ylim(0.0, 0.5)

# Hide empty subplots
for idx in range(len(BANDS), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(
    f"{SELECTED_DDF} — DRP Coadd Magnitude vs Error per band (top {TOP_N} objects)",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout()

# Save with a specific filename
fig_name = f"DRP_Obj_mag_summary_{SELECTED_DDF}.png"
plt.savefig(f"{DIR_FIGS}/{fig_name}", dpi=150, bbox_inches="tight")
plt.show()

### 9.4 Plot Source number per bands  on the six bands

In [ ]:
# --- Epochs (inputCount) per band distribution ---
fig, ax = plt.subplots(figsize=(10, 5))

# Iterate through each detected band (u, g, r, i, z, y)
for band in BANDS:
    # In DRP, 'inputCount' represents the number of visits/epochs contributing to the coadd
    col = f"{band}_inputCount"

    if col in df_obj_rich.columns:
        # Get data for objects that were actually observed in this band
        vals = df_obj_rich[col].dropna()
        vals = vals[vals > 0]  # Only consider positive counts

        if len(vals) > 0:
            ax.hist(
                vals,
                bins=50,
                alpha=0.55,
                histtype="stepfilled",
                color=BAND_COLORS.get(band, "grey"),
                label=f"{band} (med={vals.median():.0f})",
            )

# Set labels and title for DRP context
ax.set(
    xlabel="inputCount (observations per band)",
    ylabel="N Objects",
    title=f"{SELECTED_DDF} — DRP Observations per band (N={len(df_obj_rich):,})",
)

ax.legend(title="Band", fontsize=11, loc="upper right", ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save the figure with a DRP naming convention
fig_name = f"DRP_Obj_inputCount_band_{SELECTED_DDF}.png"
plt.savefig(f"{DIR_FIGS}/{fig_name}", dpi=150, bbox_inches="tight")
plt.show()

---
## 10. Load Sources

- load sources only from selected objects

In [ ]:
rich_ids = set(df_obj_rich[OBJ_ID_COL].values)
print(f"Object IDs to keep: {len(rich_ids):,}")

In [ ]:
frc_src_frames = []
# patch_id = SELECTED_PATCH
for tract_id in tract_ids:
    refs = list(
        registry.queryDatasets(
            DATASET_TYPE_FORCED,
            collections=COLLECTION,
            # dataId={"skymap": SKYMAP_NAME, "tract": tract_id, "patch": patch_id},
            dataId={"skymap": SKYMAP_NAME, "tract": tract_id},
        )
    )
    for ref in refs:
        try:
            df_tmp = to_dataframe(butler.get(ref))
            join_col = next((c for c in [OBJ_ID_COL, "objectId", "object_id"] if c in df_tmp.columns), None)
            if join_col:
                df_tmp = df_tmp[df_tmp[join_col].isin(rich_ids)]
            else:
                print(
                    f"  WARNING tract {tract_id}: join col not found. " f"Cols: {list(df_tmp.columns[:10])}"
                )
            frc_src_frames.append(df_tmp)
            print(f"  Tract {tract_id:6d} — {len(df_tmp):,} Sources")
        except Exception as exc:
            print(f"  Tract {tract_id:6d} — ERROR: {exc}")

if frc_src_frames:
    df_fsrc_rich = pd.concat(frc_src_frames, ignore_index=True)
    src_id = next(
        (c for c in ["sourceId", "source_id", "object_forced_source"] if c in df_fsrc_rich.columns), None
    )
    if src_id:
        df_fsrc_rich = df_fsrc_rich.drop_duplicates(subset=src_id)
    print(f"\nTotal Sources: {len(df_fsrc_rich):,}")
    print(f"Columns: {list(df_fsrc_rich.columns)}")
else:
    print("No Sources loaded.")
    df_fsrc_rich = pd.DataFrame()

#### Check if all columns necessary for LC are there

In [ ]:
if len(df_fsrc_rich) > 0:
    MJD_COL = next(
        (c for c in ["midPointMjdTai", "midpointMjdTai", "mjd"] if c in df_fsrc_rich.columns), None
    )
    BAND_COL = next((c for c in ["band", "filterName", "filter"] if c in df_fsrc_rich.columns), None)
    FLUX_COL = next((c for c in ["psfFlux", "psf_flux"] if c in df_fsrc_rich.columns), None)
    FERR_COL = next((c for c in ["psfFluxErr", "psf_flux_err"] if c in df_fsrc_rich.columns), None)
    print(f"MJD={MJD_COL}  band={BAND_COL}  flux={FLUX_COL}+/-{FERR_COL}")
else:
    MJD_COL = BAND_COL = FLUX_COL = FERR_COL = None

In [ ]:
if MJD_COL and FLUX_COL and BAND_COL and len(df_src_rich) > 0:
    fig, ax = plt.subplots(figsize=(16, 6))
    for band, grp in df_dia_src_rich.groupby(BAND_COL):
        ax.scatter(
            grp[MJD_COL],
            grp[FLUX_COL],
            s=1,
            alpha=0.2,
            color=BAND_COLORS.get(str(band), "grey"),
            label=str(band),
        )
    ax.set(
        xlabel="MJD (TAI)",
        ylabel="PSF flux (nJy)",
        title=f"Sources — {SELECTED_DDF} ({len(df_dia_src_rich):,})",
    )
    ax.legend(title="Band", fontsize=12, markerscale=6)
    ax.grid(True, alpha=0.3)
    add_date_top_axis(ax, n_ticks=8)
    plt.tight_layout()
    plt.savefig(f"{DIR_FIGS}/Src_scatter_{SELECTED_DDF}.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Sources scatter skipped.")

In [ ]:
del frc_src_frames
del df_fsrc_rich
gc.collect()

---
## 11. Load Forced Photometry (`object_forced_source`)

This dataset has dimensions `(skymap, tract, patch)` — there are **~100 patch refs per tract**.
Each patch file contains per-visit forced-PSF photometry for the DiaObjects in that patch.
We iterate over all patch refs, load, filter on `rich_ids`, and concatenate.

**NOTE :** `dia_object_forced_source` does not contain any time column.
It only carries columns like ('diaObjectId', 'parentObjectId', 'coord_ra', 'coord_dec', 'visit', 'detector', 'band', ...).

**Solution :** the `visitTable` retrieved from the registry in Section 2.2 maps each `visit id` (column `id`)
to its observation time (column `mjd`).  After loading all forced-photometry rows we merge this lookup
table on the `visit` column, creating a new column **`mjd_visit`** that is used as the time axis for light curves.

### 11.1 Read Forced Photometry from Butler

In [ ]:
forced_frames = []

# loop on tracts
for tract_id in tract_ids:
    # Query all patch refs for this tract (no patch constraint -> all patches)
    refs = list(
        registry.queryDatasets(
            DATASET_TYPE_FORCED,
            collections=COLLECTION,
            dataId={"skymap": SKYMAP_NAME, "tract": tract_id},
        )
    )
    print(f"Tract {tract_id:6d} — {len(refs)} patch refs")

    # loop on force photometry dia sources refs for that tract
    for idx, ref in enumerate(refs):
        patch_id = ref.dataId.get("patch", "?")

        # if patch_id != SELECTED_PATCH:
        #    continue

        try:
            obj = butler.get(ref)
            df_tmp = obj.to_pandas() if hasattr(obj, "to_pandas") else obj

            # Detect join column in this table
            join_col = next((c for c in [OBJ_ID_COL, "objectId", "object_id"] if c in df_tmp.columns), None)
            # If ID is in index (same convention as dia_object)
            if join_col is None and df_tmp.index.name in [OBJ_ID_COL, "objectId", "object_id"]:
                df_tmp = df_tmp.copy()
                df_tmp.insert(0, df_tmp.index.name, df_tmp.index)
                df_tmp = df_tmp.reset_index(drop=True)
                join_col = df_tmp.columns[0]

            if join_col:
                df_filt = df_tmp[df_tmp[join_col].isin(rich_ids)]
            else:
                print(
                    f"    Patch {patch_id}: no join col. "
                    f"idx={df_tmp.index.name!r}  cols={list(df_tmp.columns[:8])}"
                )
                df_filt = df_tmp  # keep all — will be filtered later

            if len(df_filt) > 0:
                df_filt = df_filt.copy()
                df_filt["_tract"] = tract_id
                df_filt["_patch"] = patch_id
                forced_frames.append(df_filt)

        except Exception as exc:
            print(f"    Patch {patch_id} ERROR: {exc}")

if forced_frames:
    df_forced_rich = pd.concat(forced_frames, ignore_index=True)
    print(f"\nTotal forced rows: {len(df_forced_rich):,}")
    print(f"All columns ({len(df_forced_rich.columns)}):")
    print(list(df_forced_rich.columns))
else:
    print("No forced photometry loaded.")
    df_forced_rich = pd.DataFrame()

### 11.2 Add Magnitude

In [ ]:
# --- 1. Filter for positive flux to avoid log10 errors ---
mask_pos = df_forced_rich["psfFlux"] > 0

# --- 2. Calculate psfMag ---
# Using the standard 31.4 AB zeropoint for Rubin DRP
df_forced_rich.loc[mask_pos, "psfMag"] = 31.4 - 2.5 * np.log10(df_forced_rich.loc[mask_pos, "psfFlux"])

# --- 3. Calculate psfMagErr ---
# Formula: sigma_mag = (2.5 / ln(10)) * (flux_err / flux)
# Pre-factor (2.5 / np.log(10)) is approximately 1.0857
df_forced_rich.loc[mask_pos, "psfMagErr"] = 1.0857 * (
    df_forced_rich.loc[mask_pos, "psfFluxErr"] / df_forced_rich.loc[mask_pos, "psfFlux"]
)

print(f"Calculated psfMag and psfMagErr for {mask_pos.sum():,} rows with positive flux.")

In [ ]:
df_forced_rich.head()

### 11.3 Plot on Variability

In [ ]:
def plot_variability_stats(df, mag_limit=None):
    """
    Plots the distribution of variability metrics (Sigma, MAD, IQR).

    Parameters:
    df (pd.DataFrame): The forced source dataframe (must contain 'psfMag', 'objectId', 'band').
    mag_limit (float, optional): Magnitude threshold. If None or 0, no cut is applied.
    """

    # --- 1. Aggregation per Object and per Band ---
    # We calculate the median magnitude ('mag') AND the 3 metrics
    df_stats = (
        df.groupby(["objectId", "band"])["psfMag"]
        .agg(
            [
                ("mag", "median"),
                ("sigma", "std"),
                ("sigma_mad", lambda x: 1.4826 * median_abs_deviation(x, nan_policy="omit")),
                ("sigma_iqr", lambda x: 0.7413 * (np.percentile(x, 75) - np.percentile(x, 25))),
            ]
        )
        .reset_index()
    )

    # --- 2. Apply Magnitude Cut ---
    if mag_limit and mag_limit > 0:
        df_plot = df_stats[df_stats["mag"] < mag_limit].copy()
        cut_suffix = f"mag_cut_{mag_limit}"
        title_suffix = f"(Objects with mag < {mag_limit})"
    else:
        df_plot = df_stats.copy()
        cut_suffix = "no_cut"
        title_suffix = "(All Objects)"

    print(f"Total objects/bands: {len(df_stats)}")
    print(f"Plotting {len(df_plot)} entries ({cut_suffix})")

    # --- 3. Plotting ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    metrics = [
        ("sigma", "Standard Deviation ($\sigma$)", "steelblue"),
        ("sigma_mad", "Median Absolute Deviation ($\sigma_{MAD}$)", "darkorange"),
        ("sigma_iqr", "Interquartile Range ($\sigma_{IQR}$)", "seagreen"),
    ]

    for i, (col, label, base_color) in enumerate(metrics):
        ax = axes[i]
        for band in BANDS:
            # Filter by band
            band_data = df_plot[df_plot["band"] == band][col].dropna()

            if len(band_data) > 0:
                ax.hist(
                    band_data,
                    bins=50,
                    histtype="step",
                    lw=2,
                    label=f"{band}",
                    color=BAND_COLORS.get(band, base_color),
                    alpha=0.8,
                )

        ax.set_title(f"{label}\n{title_suffix}", fontsize=13, fontweight="bold")
        ax.set_xlabel("$\sigma_{equivalent}$ [mag]", fontsize=12)
        ax.set_ylabel("N Objects", fontsize=12)
        ax.set_yscale("log")
        ax.legend(ncol=2, fontsize=10)
        ax.grid(True, alpha=0.2, ls="--")

    plt.tight_layout()
    plt.suptitle(f"Variability Statistics per Object — {SELECTED_DDF}", fontsize=16, y=1.08)

    # --- 4. Save with adapted filename ---
    fig_name = f"DRP_variability_stats_{cut_suffix}_{SELECTED_DDF}.png"
    save_path = f"{DIR_FIGS}/{fig_name}"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Saved figure to: {save_path}")
    plt.show()


# --- Example Usage ---
# plot_variability_stats(df_forced_rich, mag_limit=22)
# plot_variability_stats(df_forced_rich, mag_limit=None)

In [ ]:
def plot_variability_vs_mag(df, mag_limit=None, ymax=2.0):
    """
    Plots Variability metrics vs Median Magnitude (Scatter plots).

    Parameters:
    df (pd.DataFrame): The forced source dataframe (must contain 'psfMag', 'objectId', 'band').
    mag_limit (float, optional): Magnitude threshold. If None or 0, no cut is applied.
    """

    # --- 1. Aggregation per Object and per Band ---
    df_stats = (
        df.groupby(["objectId", "band"])["psfMag"]
        .agg(
            [
                ("mag", "median"),
                ("sigma", "std"),
                ("sigma_mad", lambda x: 1.4826 * median_abs_deviation(x, nan_policy="omit")),
                ("sigma_iqr", lambda x: 0.7413 * (np.percentile(x, 75) - np.percentile(x, 25))),
            ]
        )
        .reset_index()
    )

    # --- 2. Apply Magnitude Cut ---
    if mag_limit and mag_limit > 0:
        df_plot = df_stats[df_stats["mag"] < mag_limit].copy()
        cut_suffix = f"mag_cut_{mag_limit}"
        title_suffix = f"(mag < {mag_limit})"
    else:
        df_plot = df_stats.copy()
        cut_suffix = "no_cut"
        title_suffix = "(All)"

    print(f"Total objects/bands: {len(df_stats)}")
    print(f"Plotting {len(df_plot)} points ({cut_suffix})")

    # --- 3. Plotting ---
    fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharex=True)

    metrics = [
        ("sigma", "Standard Deviation ($\sigma$)", "steelblue"),
        ("sigma_mad", "Median Absolute Deviation ($\sigma_{MAD}$)", "darkorange"),
        ("sigma_iqr", "Interquartile Range ($\sigma_{IQR}$)", "seagreen"),
    ]

    for i, (col, label, base_color) in enumerate(metrics):
        ax = axes[i]
        for band in BANDS:
            mask = df_plot["band"] == band
            band_data = df_plot[mask]

            if len(band_data) > 0:
                ax.scatter(
                    band_data["mag"],
                    band_data[col],
                    s=5,
                    alpha=0.5,
                    label=band,
                    color=BAND_COLORS.get(band, base_color),
                    edgecolors="none",
                )

        ax.set_title(f"{label} vs Magnitude\n{title_suffix}", fontsize=14, fontweight="bold")
        ax.set_xlabel("Median psfMag (AB)", fontsize=12)
        ax.set_ylabel(r"$\sigma_{equivalent}$ [mag]", fontsize=12)

        # Set limits as requested
        ax.set_ylim(0.0, ymax)

        # Invert X-axis: Bright objects to the left
        # ax.invert_xaxis()

        ax.legend(title="Band", ncol=2, markerscale=3, fontsize=9, loc="upper left")
        ax.grid(True, which="both", alpha=0.2)

    plt.tight_layout()
    plt.suptitle(f"Variability vs Magnitude — {SELECTED_DDF}", fontsize=18, y=1.05)

    # --- 4. Save and Show ---
    fig_name = f"DRP_variability_vs_mag_{cut_suffix}_{SELECTED_DDF}.png"
    save_path = f"{DIR_FIGS}/{fig_name}"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Saved scatter plot to: {save_path}")
    plt.show()


# --- Example Usage ---
# plot_variability_vs_mag(df_forced_rich, mag_limit=23)

In [ ]:
plot_variability_vs_mag(df_forced_rich, mag_limit=None)

In [ ]:
plot_variability_stats(df_forced_rich, mag_limit=None)

In [ ]:
plot_variability_stats(df_forced_rich, mag_limit=22)

In [ ]:
plot_variability_stats(df_forced_rich, mag_limit=23)

In [ ]:
plot_variability_stats(df_forced_rich, mag_limit=24)

### 11.4 Associate Visit and MJD

In [ ]:
FULLFILENAME_VISITS = os.path.join(OUTPUT_DIR, FILENAME_VISITS)
print(f"Visit Filename : {FULLFILENAME_VISITS} ")

In [ ]:
df_visitTable = load_visits_file(FULLFILENAME_VISITS)

In [ ]:
display(df_visitTable.head())

In [ ]:
if df_visitTable is not None and not df_visitTable.empty:
    # 1. Identify columns
    # Note: fixed typo in your original code where you referenced df_visits instead of df_visitTable
    visit_col = "visit" if "visit" in df_visitTable.columns else "id"
    mjd_col = "expMidptMJD" if "expMidptMJD" in df_visitTable.columns else "mjd"

    try:
        # 2. Extract and Rename
        visit_mjd_lookup = df_visitTable[[visit_col, mjd_col]].copy()
        visit_mjd_lookup = visit_mjd_lookup.rename(columns={visit_col: "visit", mjd_col: "mjd_visit"})

        # 3. Clean duplicates
        visit_mjd_lookup = visit_mjd_lookup.drop_duplicates(subset="visit")

        # REMOVED: index naming and reset_index() because "visit" is already a column.
        # If you want to ensure the index is just numbers:
        visit_mjd_lookup = visit_mjd_lookup.reset_index(drop=True)

        print(f"Lookup table created with {len(visit_mjd_lookup)} unique visits.")

        del df_visitTable
        import gc

        gc.collect()

    except KeyError as e:
        print(f"Column error: {e}. Available columns: {list(df_visitTable.columns)}")
else:
    print("df_visitTable is None or empty.")

In [ ]:
display(visit_mjd_lookup.head())

In [ ]:
# --- 1. Perform the Merge ---
# We use a left join on 'visit' to bring 'mjd_visit' into our main dataframe
df_forced_rich = df_forced_rich.merge(visit_mjd_lookup[["visit", "mjd_visit"]], on="visit", how="left")

# --- 2. Verification ---
# Check if we have any NaN values (visits that were in forced_rich but missing from visitTable)
n_missing = df_forced_rich["mjd_visit"].isna().sum()

if n_missing > 0:
    print(f"WARNING: {n_missing} sources do not have a corresponding MJD.")
else:
    print("Success: All sources now have an associated MJD.")

# --- 3. Preview the result ---
print("\nPreview of the updated DataFrame:")
display(df_forced_rich[["objectId", "visit", "band", "psfMag", "mjd_visit"]].head())

### 11.5 Plot light curves

In [ ]:
def plot_object_lightcurves(df, band, n_curves=5, min_obs=10, mag_limit=None):
    # 1. Filter by band and valid MJD
    df_band = df[(df["band"] == band) & (df["mjd_visit"].notna())].copy()

    # 2. Get valid object IDs
    obj_stats = df_band.groupby("objectId").agg(n_obs=("psfMag", "count"), median_mag=("psfMag", "median"))

    valid_mask = obj_stats["n_obs"] >= min_obs
    if mag_limit:
        valid_mask &= obj_stats["median_mag"] < mag_limit

    valid_ids = obj_stats[valid_mask].index.tolist()

    if not valid_ids:
        print(f"No objects found in band '{band}' matching the criteria.")
        return

    selected_ids = valid_ids[:n_curves]
    n_actual = len(selected_ids)

    fig, axes = plt.subplots(n_actual, 1, figsize=(12, 2.5 * n_actual), sharex=True)
    if n_actual == 1:
        axes = [axes]

    for i, obj_id in enumerate(selected_ids):
        ax = axes[i]

        # Filter for positive flux and specific object
        obj_data = df_band[(df_band["objectId"] == obj_id) & (df_band["psfFlux"] > 0)].sort_values(
            "mjd_visit"
        )

        if obj_data.empty:
            continue

        # --- Statistics Calculation ---
        mags = obj_data["psfMag"]
        mag_errs = obj_data["psfMagErr"]
        median_val = mags.median()
        std_val = mags.std()
        mad_sigma = 1.4826 * median_abs_deviation(mags, nan_policy="omit")
        iqr_sigma = 0.7413 * (np.percentile(mags, 75) - np.percentile(mags, 25))

        # Calculate Magnitude Error
        # mag_err = 1.0857 * (obj_data['psfFluxErr'] / obj_data['psfFlux'])

        # --- Plotting ---
        label_str = (
            f"Obj: {obj_id}\n"
            f"$\sigma$={std_val:.3f} | "
            f"$\sigma_{{MAD}}$={mad_sigma:.3f} | "
            f"$\sigma_{{IQR}}$={iqr_sigma:.3f}"
        )

        ax.errorbar(
            obj_data["mjd_visit"],
            mags,
            yerr=mag_errs,
            fmt="o",
            markersize=4,
            capsize=2,
            elinewidth=1,
            color=BAND_COLORS.get(band, "black"),
            alpha=0.7,
            label=label_str,
        )

        # Draw Median line and +/- 5 mmag (0.005 mag)
        ax.axhline(median_val, color="red", linestyle="-", alpha=0.6, lw=1)
        ax.axhline(median_val + 0.005, color="red", linestyle="--", alpha=0.3, lw=0.8)
        ax.axhline(median_val - 0.005, color="red", linestyle="--", alpha=0.3, lw=0.8)

        # --- Y-Axis Centering ---
        # We center the range around the median.
        # Using a window of +/- 0.2 mag or 4*sigma (whichever is larger)
        y_range = max(0.15, 4 * mad_sigma)
        ax.set_ylim(median_val + y_range, median_val - y_range)  # Inverted: higher mag at bottom

        ax.set_ylabel(f"psfMag ({band})", fontsize=10)
        ax.legend(loc="upper right", fontsize=8, frameon=True, facecolor="white", framealpha=0.9)
        ax.grid(True, alpha=0.5, ls="-")

    axes[-1].set_xlabel("MJD (Modified Julian Date)", fontsize=11)
    add_date_top_axis(axes[0])

    plt.tight_layout()
    plt.show()

In [ ]:
plot_object_lightcurves(df_forced_rich, band="r", n_curves=10, min_obs=25, mag_limit=22.5)

### 11.6  Light curves from specific selections

In [ ]:
def select_variable_objects(
    df,
    band,
    min_obs=20,
    mag_limit=22.5,
    sigma_min=None,
    sigma_max=None,
    sigma_mad_min=None,
    sigma_iqr_max=None,
):
    """
    Returns a list of objectIds matching specific variability and quality criteria.
    """
    # Filter by band and valid detections
    mask = (df["band"] == band) & (df["psfFlux"] > 0) & (df["mjd_visit"].notna())
    df_temp = df[mask].copy()

    # Calculate stats per object
    stats = (
        df_temp.groupby("objectId")["psfMag"]
        .agg(
            [
                ("n_obs", "count"),
                ("mag", "median"),
                ("sigma", "std"),
                ("sigma_mad", lambda x: 1.4826 * median_abs_deviation(x, nan_policy="omit")),
                ("sigma_iqr", lambda x: 0.7413 * (np.percentile(x, 75) - np.percentile(x, 25))),
            ]
        )
        .reset_index()
    )

    # --- Apply Filters ---
    # Basic quality cuts
    mask_filter = stats["n_obs"] >= min_obs
    if mag_limit:
        mask_filter &= stats["mag"] < mag_limit

    # Variability cuts (Note: 10 mmag = 0.010)
    if sigma_min is not None:
        mask_filter &= stats["sigma"] > sigma_min
    if sigma_max is not None:
        mask_filter &= stats["sigma"] < sigma_max
    if sigma_mad_min is not None:
        mask_filter &= stats["sigma_mad"] > sigma_mad_min
    if sigma_iqr_max is not None:
        mask_filter &= stats["sigma_iqr"] < sigma_iqr_max

    selected_ids = stats.loc[mask_filter, "objectId"].tolist()
    print(f"Found {len(selected_ids)} objects matching criteria in band {band}.")

    return selected_ids

In [ ]:
def plot_specific_objects(df, object_ids, band, n_curves=5):
    """
    Plots light curves for a specific list of object IDs with band-specific colors.
    """
    if not object_ids:
        print("No IDs provided to plot.")
        return

    selected_ids = object_ids[:n_curves]
    n_actual = len(selected_ids)

    # Get color for the current band, default to black if not found
    plot_color = BAND_COLORS.get(band, "black")

    fig, axes = plt.subplots(n_actual, 1, figsize=(12, 2.5 * n_actual), sharex=True)
    if n_actual == 1:
        axes = [axes]

    for i, obj_id in enumerate(selected_ids):
        ax = axes[i]

        # Filter data for this object and band
        obj_data = df[(df["objectId"] == obj_id) & (df["band"] == band) & (df["psfFlux"] > 0)].sort_values(
            "mjd_visit"
        )

        if obj_data.empty:
            ax.text(0.5, 0.5, f"No data for {obj_id} in band {band}", transform=ax.transAxes, ha="center")
            continue

        mags = obj_data["psfMag"]
        mag_errs = obj_data["psfMagErr"]
        median_val = mags.median()

        # Stats for the legend
        s_std = mags.std()
        s_mad = 1.4826 * median_abs_deviation(mags, nan_policy="omit")
        s_iqr = 0.7413 * (np.percentile(mags, 75) - np.percentile(mags, 25))

        # Magnitude error propagation
        # mag_err = 1.0857 * (obj_data['psfFluxErr'] / obj_data['psfFlux'])

        # Legend label including Band name and Stats
        legend_label = (
            f"ID: {obj_id} (band: {band})\n"
            f"$\sigma$={s_std:.3f} | $\sigma_{{MAD}}$={s_mad:.3f} | $\sigma_{{IQR}}$={s_iqr:.3f}"
        )

        # Plotting with band color
        ax.errorbar(
            obj_data["mjd_visit"],
            mags,
            yerr=mag_errs,
            fmt="o",
            markersize=4,
            color=plot_color,
            alpha=0.7,
            capsize=2,
            elinewidth=1,
            label=legend_label,
        )

        # Median and +/- 5mmag lines
        ax.axhline(median_val, color="red", lw=1.5, alpha=0.8)
        ax.axhline(median_val + 0.005, color="red", ls=":", alpha=0.4)
        ax.axhline(median_val - 0.005, color="red", ls=":", alpha=0.4)

        # Center Y-axis (Standard astronomical convention: lower mag at top)
        # Using a dynamic range based on the MAD to avoid showing just noise
        y_range = max(0.05, 6 * s_mad)
        ax.set_ylim(median_val + y_range, median_val - y_range)

        ax.set_ylabel(f"psfMag ({band})", fontsize=10)
        ax.legend(loc="upper right", fontsize=9, frameon=True, shadow=True)
        ax.grid(True, alpha=0.5, ls="--")

    axes[-1].set_xlabel("MJD (Days)", fontsize=11)

    # Add the top date axis (using your previously defined helper)
    add_date_top_axis(axes[0])

    plt.tight_layout()
    plt.show()

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# 1. Identify the targets
my_targets = select_variable_objects(
    df_forced_rich,
    band="r",
    mag_limit=22.0,
    sigma_max=0.050,  # < 10 mmag
    # sigma_iqr_max=0.050 # < 50 mmag
)

# 2. Plot the first few
plot_specific_objects(df_forced_rich, my_targets, band="r", n_curves=10)

In [ ]:
# 1. Identify the targets
my_targets = select_variable_objects(
    df_forced_rich,
    band="z",
    mag_limit=22.0,
    sigma_min=0.050,  # < 10 mmag
    # sigma_iqr_max=0.050 # < 50 mmag
)

# 2. Plot the first few
plot_specific_objects(df_forced_rich, my_targets, band="z", n_curves=10)

In [ ]:
# 1. Identify the targets
my_targets = select_variable_objects(
    df_forced_rich,
    band="y",
    mag_limit=23,
    sigma_min=0.050,  # < 10 mmag
    # sigma_iqr_max=0.050 # < 50 mmag
)

# 2. Plot the first few
plot_specific_objects(df_forced_rich, my_targets, band="y", n_curves=50)